# Limpieza de `data_raw.csv`

Notebook de limpieza del dataset crudo del NASA Exoplanet Archive (`data/data_raw.csv`).

**El fichero original (`data_raw.csv`) NO se modifica**: el notebook trabaja sobre una copia en memoria y exporta el resultado a un fichero nuevo (`data/data_raw_processed.csv`).

Pasos:
1. Leer el CSV saltando los primeros **96 registros** (metadatos comentados con `#`). El **registro 97** contiene los nombres de columna.
2. Quedarnos solo con las **18 columnas definitivas**.
3. Eliminar **registros duplicados**.
4. Exportar el resultado a `data/data_raw_processed.csv`.

In [1]:
import pandas as pd
from pathlib import Path

## 1. Carga del CSV

Saltamos los primeros 96 registros (cabecera con `#` que describe cada columna). Pandas tomará el registro 97 como cabecera real.

In [2]:
RAW_PATH = Path('..') / 'data' / 'data_raw.csv'

# Leemos el CSV original. El DataFrame resultante es una copia en memoria;
# nada de lo que hagamos a continuación toca el fichero original.
df = pd.read_csv(RAW_PATH, skiprows=96, low_memory=False).copy()

print(f'Filas: {len(df):,} | Columnas: {df.shape[1]}')
df.head()

Filas: 39,852 | Columnas: 92


,pl_name,hostname,default_flag,sy_snum,sy_pnum,discoverymethod,disc_year,disc_facility,soltype,pl_controv_flag,...,sy_vmagerr2,sy_kmag,sy_kmagerr1,sy_kmagerr2,sy_gaiamag,sy_gaiamagerr1,sy_gaiamagerr2,rowupdate,pl_pubdate,releasedate
0,11 Com b,11 Com,1,2,1,Radial Velocity,2007.0,Xinglong Station,Published Confirmed,0,...,-0.023,2.282,0.346,-0.346,4.44038,0.003848,-0.003848,2023-09-19,2023-12,2023-09-19
1,11 Com b,11 Com,0,2,1,Radial Velocity,2007.0,Xinglong Station,Published Confirmed,0,...,-0.023,2.282,0.346,-0.346,4.44038,0.003848,-0.003848,2014-07-23,2011-08,2014-07-23
2,11 Com b,11 Com,0,2,1,Radial Velocity,2007.0,Xinglong Station,Published Confirmed,0,...,-0.023,2.282,0.346,-0.346,4.44038,0.003848,-0.003848,2014-05-14,2008-01,2014-05-14
3,11 UMi b,11 UMi,0,1,1,Radial Velocity,2009.0,Thueringer Landessternwarte Tautenburg,Published Confirmed,0,...,-0.005,1.939,0.270,-0.270,4.56216,0.003903,-0.003903,2018-04-25,2011-08,2014-07-23
4,11 UMi b,11 UMi,0,1,1,Radial Velocity,2009.0,Thueringer Landessternwarte Tautenburg,Published Confirmed,0,...,-0.005,1.939,0.270,-0.270,4.56216,0.003903,-0.003903,2018-04-25,2009-10,2014-05-14


## 2. Selección de columnas definitivas

| Columna original | Significado |
|---|---|
| `pl_name` | Nombre planeta |
| `hostname` | Estrella (host name) |
| `sy_snum` | Número de estrellas |
| `sy_pnum` | Número de planetas |
| `discoverymethod` | Método de descubrimiento |
| `disc_year` | Año |
| `disc_facility` | Centro de descubrimiento |
| `soltype` | Solution Type |
| `pl_controv_flag` | Controversial flag |
| `pl_orbper` | Días de periodo orbital |
| `pl_rade` | Planet radius (Earth) |
| `pl_bmasse` | Masa del planeta (Earth) |
| `pl_eqt` | Equilibrium temperature |
| `st_spectype` | Spectral type |
| `st_teff` | Stellar effective temperature |
| `st_rad` | Stellar radius (Solar) |
| `st_mass` | Stellar mass |
| `sy_dist` | Distance |

In [3]:
columnas_definitivas = [
    'pl_name',
    'hostname',
    'sy_snum',
    'sy_pnum',
    'discoverymethod',
    'disc_year',
    'disc_facility',
    'soltype',
    'pl_controv_flag',
    'pl_orbper',
    'pl_rade',
    'pl_bmasse',
    'pl_eqt',
    'st_spectype',
    'st_teff',
    'st_rad',
    'st_mass',
    'sy_dist',
]

df = df[columnas_definitivas]
print(f'Filas: {len(df):,} | Columnas: {df.shape[1]}')
df.head()

Filas: 39,852 | Columnas: 18


,pl_name,hostname,sy_snum,sy_pnum,discoverymethod,disc_year,disc_facility,soltype,pl_controv_flag,pl_orbper,pl_rade,pl_bmasse,pl_eqt,st_spectype,st_teff,st_rad,st_mass,sy_dist
0,11 Com b,11 Com,2,1,Radial Velocity,2007.0,Xinglong Station,Published Confirmed,0,323.21,NaN,4914.898486,NaN,G8 III,4874.0,13.76,2.09,93.1846
1,11 Com b,11 Com,2,1,Radial Velocity,2007.0,Xinglong Station,Published Confirmed,0,NaN,NaN,5434.700000,NaN,NaN,NaN,NaN,2.60,93.1846
2,11 Com b,11 Com,2,1,Radial Velocity,2007.0,Xinglong Station,Published Confirmed,0,326.03,NaN,6165.600000,NaN,G8 III,4742.0,19.00,2.70,93.1846
3,11 UMi b,11 UMi,1,1,Radial Velocity,2009.0,Thueringer Landessternwarte Tautenburg,Published Confirmed,0,NaN,NaN,3432.400000,NaN,NaN,NaN,NaN,1.70,125.3210
4,11 UMi b,11 UMi,1,1,Radial Velocity,2009.0,Thueringer Landessternwarte Tautenburg,Published Confirmed,0,516.22,NaN,3337.070000,NaN,K4 III,4340.0,24.08,1.80,125.3210


## 3. Eliminación de duplicados

In [4]:
duplicados = df.duplicated().sum()
print(f'Duplicados detectados: {duplicados:,}')

df = df.drop_duplicates().reset_index(drop=True)
print(f'Filas tras eliminar duplicados: {len(df):,}')

Duplicados detectados: 148
Filas tras eliminar duplicados: 39,704


## 4. Guardado del dataset procesado

Exportamos el resultado a `data/data_raw_processed.csv`.

In [5]:
OUTPUT_PATH = Path('..') / 'data' / 'data_raw_processed.csv'
df.to_csv(OUTPUT_PATH, index=False)
print(f'Guardado en: {OUTPUT_PATH.resolve()}')

Guardado en: /Users/sitomachucas/Documents/etl_clase_07_05-1/data/data_raw_processed.csv
